# Examen de Recuperación — Servicio RAG para Information Retrieval

**ICCD753 Recuperación de Información 2026-A · Prof. Iván Carrera · EPN-FIS**

Estudiante: *(coloca tu nombre)*

Repositorio: *(URL del repositorio)*

URL del servicio desplegado: **`https://prone-snowiness-reviving.ngrok-free.dev`**

---

Este notebook documenta el desarrollo completo del servicio web RAG exigido por el examen de recuperación. Cada celda ejecuta una parte del pipeline; los artefactos persistentes (índice Chroma, *manifest*) se almacenan en `data/` para que el servicio pueda arrancar en frío sin reindexar.

## Índice

A. Configuración del entorno y dependencias
B. Adquisición y registro bibliográfico del corpus
C. Procesamiento del corpus (extracción, limpieza, *chunking*)
D. Indexación en base vectorial
E. *Retrieval* denso + re-ranking con cross-encoder
F. Generación de respuestas fundamentadas
G. Servicio web (FastAPI)
H. Despliegue y ejemplos de consumo
I. Códigos de estado HTTP
J. Limitaciones y decisiones de diseño

## A. Configuración del entorno

Instala las dependencias en el entorno actual. Si ejecutas el notebook en Hugging Face Spaces, el `Dockerfile` ya las instala; este paso es para ejecuciones locales.

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path('.').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
os.environ.setdefault('DATA_DIR', str(PROJECT_ROOT / 'data'))
os.environ.setdefault('CHROMA_DIR', str(PROJECT_ROOT / 'data' / 'chroma'))
os.environ.setdefault('CHROMA_COLLECTION', 'ir_corpus')
os.environ.setdefault('LOG_LEVEL', 'INFO')

# En HF Spaces, define LLM_API_KEY como secreto. En local, puedes usar:
# os.environ['LLM_API_KEY'] = 'gsk_...'

from ir_rag.config import Settings
settings = Settings.from_env()
print('Configuración cargada:')
print(f'  corpus_dir     = {settings.corpus_dir}')
print(f'  chroma_dir     = {settings.chroma_dir}')
print(f'  embedding      = {settings.embedding_model}')
print(f'  reranker       = {settings.reranker_model}')
print(f'  LLM model      = {settings.llm_model}')
print(f'  LLM key set?   = {bool(settings.llm_api_key)}')

## B. Adquisición del corpus

### B.1 Bibliografía obligatoria

1. **Baeza-Yates & Ribeiro-Neto — *Modern Information Retrieval*** (1999). **Protegido por derechos de autor**: descárgalo de tu biblioteca o copia personal y colócalo en `corpus/baeza-yates-modern-ir.pdf`. Si la edición tiene metadatos distintos, ajusta `CORPUS_REGISTRY` en `src/ir_rag/corpus.py`.
2. **Manning, Raghavan & Schütze — *Introduction to Information Retrieval*** (2009). Acceso abierto desde Stanford NLP.

In [ ]:
# Descarga los PDFs de acceso abierto. Si ya están en corpus/, se omiten.
!python3 scripts/download_corpus.py

In [ ]:
from ir_rag.corpus import discover_sources, CORPUS_REGISTRY

print('Bibliografía registrada en el sistema:')
for filename, meta in CORPUS_REGISTRY.items():
    print(f'  - {meta["doc_id"]:>22s}  [{meta["kind"]:>7s}]  {filename}')

sources = discover_sources(settings.corpus_dir, settings.articles_dir)
print(f'\nFuentes encontradas en disco: {len(sources)}')
for s in sources:
    print(f'  - {s.doc_id:>22s}  {Path(s.file_path).relative_to(PROJECT_ROOT)}')

## C. Procesamiento del corpus

Cada PDF pasa por:

1. Extracción de texto página por página (PyMuPDF).
2. Detección y eliminación de encabezados/pies repetidos (umbral relativo al total de páginas).
3. Identificación de capítulos y secciones por número + tamaño de fuente + negrita.
4. División en *chunks* de `CHUNK_SIZE` caracteres con solapamiento.
5. Asignación de ID único `{doc_id}:p{page}:c{idx:04d}-{slug}`.

In [ ]:
from ir_rag.corpus import extract_pages

# Vista rápida del primer libro obligatorio que encontremos en disco.
manning = next(s for s in sources if s.doc_id == 'manning-2009')
pages = extract_pages(manning)
print(f'"{manning.title}" → {len(pages)} páginas tras limpieza')

# Mostrar la primera página con encabezado/sección detectados
for i, p in enumerate(pages[:60]):
    if p.chapter:
        print(f'  p.{p.page_number}: chapter = {p.chapter!r}')
        if p.section:
            print(f'           section = {p.section!r}')
        break

In [ ]:
from ir_rag.corpus import build_corpus

chunks, stats = build_corpus(
    sources,
    chunk_size=settings.chunk_size,
    chunk_overlap=settings.chunk_overlap,
    min_chars=settings.min_chunk_chars,
)

print(f'Total chunks: {len(chunks)}')
for doc_id, n in stats['per_source'].items():
    print(f'  {doc_id:>22s}: {n:>5d} chunks')
print(f'Fallos: {stats["failed"]}')

In [ ]:
# Inspección de un chunk real (debe incluir página y sección).
for c in chunks:
    if c.doc_id == 'manning-2009' and c.section:
        print(f'chunk_id    = {c.chunk_id}')
        print(f'doc_id      = {c.doc_id}')
        print(f'title       = {c.title}')
        print(f'authors     = {", ".join(c.authors)}')
        print(f'chapter     = {c.chapter}')
        print(f'section     = {c.section}')
        print(f'page_start  = {c.page_start}')
        print(f'page_end    = {c.page_end}')
        print(f'--- text ---')
        print(c.text[:600] + ('...' if len(c.text) > 600 else ''))
        break

## D. Indexación en base vectorial

Embeddings con `sentence-transformers/all-MiniLM-L6-v2` (normalizados) y Chroma persistente con cosine similarity.

In [ ]:
from ir_rag.indexing import build_index

def progress(done: int, total: int) -> None:
    print(f'  embedding {done:>6d}/{total:<6d} chunks', flush=True)

manifest = build_index(settings, progress=progress)
print('\nManifest:')
for key, value in manifest['stats'].items():
    print(f'  {key:>20s}: {value}')

## E. Retrieval denso + re-ranking

Cada consulta:
1. Se codifica con el mismo modelo de embeddings.
2. Se buscan los `RETRIEVAL_CANDIDATES` (20) vecinos más cercanos.
3. Se reordenan con el cross-encoder `ms-marco-MiniLM-L-6-v2`.
4. Se eligen los `TOP_K` (5) más relevantes, con diversidad por documento.

In [ ]:
from ir_rag.retriever import Retriever

retriever = Retriever(settings)
queries = [
    '¿Qué es BM25 y cómo funciona?',
    'How does dense passage retrieval differ from BM25?',
    'Explica la diferencia entre precisión y exhaustividad.',
    'What is PageRank and how is it used in web search?',
]

for q in queries:
    outcome = retriever.retrieve(q)
    print(f'\nQ: {q}')
    print(f'  insufficient = {outcome.insufficient}')
    for ev in outcome.evidence[:2]:
        print(f'  [{ev.evidence_id}] {ev.title[:45]:45s} sem={ev.semantic_score:.2f} rerank={ev.rerank_score:.2f}')
        print(f'      chunk={ev.chunk_id}')
        print(f'      p.{ev.page_start} {ev.chapter or "-"} / {ev.section or "-"}')

## F. Generación de respuestas

El LLM recibe únicamente la evidencia del corpus y debe citar los identificadores `[E1]…[E5]`. Si la evidencia es insuficiente, devuelve `[INSUFFICIENT_CONTEXT]` y el servicio responde con un mensaje explícito sin invocar al LLM.

In [ ]:
from ir_rag.rag import RAGPipeline, extract_question
from ir_rag.models import AnswerRequest

pipeline = RAGPipeline(settings)

# Markdown típico de un examen: encabezado, negrita y enlace.
md_q = (
    '## Pregunta\n\n'
    '¿Cuáles son los **componentes principales** del modelo BM25 y '
    'cómo se relacionan con [Information Retrieval](https://en.wikipedia.org/wiki/Information_retrieval)?'
)
print('Pregunta Markdown:')
print(md_q)
print('\nPregunta limpia (extract_question):')
print(extract_question(md_q))

In [ ]:
# Solo ejecuta si hay clave del LLM configurada (de lo contrario se lanza 503).
if settings.llm_api_key:
    response = pipeline.answer(AnswerRequest(question=md_q))
    print('Pregunta :', response.question)
    print('\nRespuesta :')
    print(response.answer)
    print('\nReferencias :')
    for ref in response.references:
        print(' ', ref)
    print(f'\nRetrieval_ms = {response.retrieval_ms:.1f}, Generation_ms = {response.generation_ms:.1f}')
    print(f'Insufficient = {response.insufficient}')
else:
    print('LLM_API_KEY no configurada — define la clave antes de generar respuestas.')

## G. Servicio web (FastAPI)

El servicio expone `POST /answer` y mantiene compatibilidad con Postman/curl. Arranca con `uvicorn ir_rag.api:app --host 0.0.0.0 --port 7860`.

In [ ]:
from fastapi.testclient import TestClient
from ir_rag.api import create_app

app = create_app(settings)
client = TestClient(app)

print('GET /')
r = client.get('/')
print(f'  status={r.status_code} name={r.json()["name"]}')

print('GET /health')
r = client.get('/health')
print(f'  status={r.status_code} chunks={r.json().get("chunks")}')

print('POST /answer (pregunta vacía → 400)')
r = client.post('/answer', json={'question': ''})
print(f'  status={r.status_code} detail={r.json()["detail"]}')

print('POST /answer (markdown sin pregunta → 422)')
r = client.post('/answer', json={'question': '```\ncode\n```'})
print(f'  status={r.status_code} detail={r.json()["detail"]}')

print('GET /ruta-inexistente → 404')
r = client.get('/ruta-inexistente')
print(f'  status={r.status_code} detail={r.json()["detail"]}')

## H. Despliegue y ejemplos de consumo

### H.1 URL del servicio

Tras desplegar en Hugging Face Spaces, Render u otra plataforma con HTTPS automático, pega aquí la URL pública.

**URL HTTPS**: `https://<usuario>-<space>.hf.space` *(reemplaza)*

### H.2 Consumir el servicio con curl

```bash
curl -X POST https://<HOST>/answer \
     -H 'Content-Type: application/json' \
     -d '{"question": "## ¿Qué es BM25?\n\nExplica el algoritmo y sus variantes."}'
```

### H.3 Consumir el servicio con Postman

1. Crea una nueva *request* `POST` apuntando a `{{base_url}}/answer`.
2. En *Body → raw → JSON*: `{"question": "## ¿Qué es BM25?"}`.
3. Pulsa *Send* y revisa la pestaña *Body*.

### H.4 Consumir el servicio desde Python

Ejecuta la celda siguiente reemplazando `<HOST>` por tu URL.

In [ ]:
import json
import urllib.request

HOST = 'prone-snowiness-reviving.ngrok-free.dev'   # p. ej. 'alexandor31-ir-rag.hf.space'
QUESTION = '## ¿Cómo funciona el algoritmo BM25?\n\nExplica el modelo.'

if HOST.startswith('<'):
    print('Edita la variable HOST antes de ejecutar esta celda.')
else:
    req = urllib.request.Request(
        f'https://' + HOST + '/answer',
        data=json.dumps({'question': QUESTION}).encode('utf-8'),
        headers={'Content-Type': 'application/json'},
        method='POST',
    )
    with urllib.request.urlopen(req, timeout=60) as resp:
        body = json.loads(resp.read().decode('utf-8'))
    print(f'HTTP {resp.status}')
    print(json.dumps(body, indent=2, ensure_ascii=False)[:2000])

## I. Códigos de estado HTTP

| Código | Significado |
|--------|-------------|
| 200 | Solicitud procesada correctamente. |
| 400 | Solicitud inválida o ausencia del campo `question`. |
| 404 | Endpoint o recurso no encontrado. |
| 422 | La solicitud es válida pero no fue posible identificar preguntas en el contenido recibido. |
| 500 | Error interno del servidor. |
| 503 | El modelo, la base vectorial o algún componente requerido no se encuentra disponible. |

Los códigos 503 se devuelven cuando (a) la clave del LLM no está configurada, (b) el endpoint LLM devolvió un error HTTP, (c) el índice Chroma no responde o está vacío.

## J. Limitaciones y decisiones de diseño

- **Baeza-Yates & Ribeiro-Neto**: libro protegido por derechos. El usuario debe colocar su copia en `corpus/baeza-yates-modern-ir.pdf` y verificar los metadatos en `CORPUS_REGISTRY`.
- **Sin contenido LLM en el corpus**: solo se indexa el texto extraído del PDF. Los campos `authors`, `year`, etc. se usan únicamente como etiquetas.
- **Re-ranking**: el cross-encoder `ms-marco-MiniLM-L-6-v2` es ligero y multilingüe. Si necesitas más precisión, sustitúyelo por `cross-encoder/ms-marco-MiniLM-L-12-v2` o un modelo mayor vía `RERANKER_MODEL`.
- **Modelo LLM por defecto**: `llama-3.3-70b-versatile` en Groq (gratuito). Para producción considera `gpt-4o-mini`, `claude-3-5-haiku` o un modelo local vía Ollama ajustando `LLM_API_BASE`.
- **Persistencia del índice**: Chroma se guarda en `data/chroma/`. Si despliegas en HF Spaces con `AUTO_BUILD_INDEX=true`, la primera ejecución construye el índice desde cero; para entregas definitivas pre-construye el índice en local y monta `data/chroma/` como almacenamiento persistente (o súbelo al Space como archivo).
- **HTTPS**: lo gestiona la plataforma (HF Spaces, Render, etc.). Si despliegas en un VPS, configura Caddy o Nginx + Let's Encrypt.